# Google API config

# Compare_and_Update_site_roster

<a href="https://colab.research.google.com/github/PeaceAndLongLife/Analysis-Colab/blob/Development/notebooks/Combine_roster_info_from_excel_files_in_a_folder_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ## Mount google drive and read in the SERVICE_ACCOUNT_FILE  {"form-width":"20%"}

# @markdown ---
# @markdown
# @markdown The `SERVICE_ACCOUNT_FILE` path is stored as a secret in google colab. If you do not have this sotred on your colab, contact the Admin: Travis Kregear at tkregear@pdx.edu

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# load SERVICE_ACCOUNT_FILE
from google.colab import userdata

SERVICE_ACCOUNT_FILE = userdata.get('SERVICE_ACCOUNT_FILE')

In [ ]:
# @title ## Install pyDrive  {"form-width":"20%"}
# @markdown
# @markdown ---
# @markdown
# @markdown Thie is necessary if using the Google API to call files by their file_id
# !pip install PyDrive

In [ ]:
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseUpload # Import MediaIoBaseUpload
import pandas as pd
import io

### Google Docs client manager
class GoogleDocumentManager():
    def __init__(self,
                 service_account_file=SERVICE_ACCOUNT_FILE,
                 document_id=None,
                 scopes=[
                    'https://www.googleapis.com/auth/drive'
                    ]
                 ):
        self.service_account_file = service_account_file
        self.document_id = document_id
        self.scopes = scopes

    # Helper to build drive service
    def _build_drive_service(self):
        credentials = service_account.Credentials.from_service_account_file(
                self.service_account_file,
                scopes=self.scopes)
        return build('drive', 'v3', credentials=credentials)

    # read Document
    def read_file(self, document_id, file_type='csv'):

        try:
            drive_service = self._build_drive_service()
            print("Google Drive API service built successfully.")

            # Download the file content
            request = drive_service.files().get_media(fileId=document_id)
            file_content = request.execute()

            # Read file into DataFrame based on file_type
            if file_type.lower() == 'csv':
                df = pd.read_csv(io.BytesIO(file_content))
            elif file_type.lower() == 'xlsx':
                df = pd.read_excel(io.BytesIO(file_content))
            return df

        # Raise errors is necessary
        except HttpError as error:
            print(f"An HTTP error occurred: {error}")
            return None
        except FileNotFoundError:
            print(f"Error: Service account key file not found at '{self.service_account_file}'")
            print("Please ensure the path to your service account key is correct.")
            return None
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return None

    def list_files_in_folder(self, folder_id, file_mime_type=None):
        """
        Lists files in a given Google Drive folder.

        Args:
            folder_id (str): The ID of the Google Drive folder.
            file_mime_type (str, optional): The MIME type of the files to filter by (e.g., 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' for XLSX).

        Returns:
            list: A list of dictionaries, each containing 'id' and 'name' of the files.
        """
        try:
            drive_service = self._build_drive_service()
            print(f"Google Drive API service built successfully for listing files in folder {folder_id}.")

            query = f"'{folder_id}' in parents and trashed = false"
            if file_mime_type:
                query += f" and mimeType = '{file_mime_type}'"

            results = drive_service.files().list(
                q=query,
                fields="nextPageToken, files(id, name)",
                supportsAllDrives=True,
                includeItemsFromAllDrives=True
            ).execute()
            items = results.get('files', [])

            if not items:
                print(f"No files found in folder {folder_id} matching criteria.")
                return []
            else:
                print(f"Found {len(items)} files in folder {folder_id}.")
                return items

        except HttpError as error:
            print(f"An HTTP error occurred during file listing: {error}")
            return []
        except FileNotFoundError:
            print(f"Error: Service account key file not found at '{self.service_account_file}'")
            print("Please ensure the path to your service account key is correct.")
            return []
        except Exception as e:
            print(f"An unexpected error occurred during file listing: {e}")
            return []


    def upload_dataframe_to_drive(self, df_to_upload, filename, folder_id, file_type='csv'):
        if folder_id == 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE':
            print(f"\nWarning: Please replace 'YOUR_GOOGLE_DRIVE_FOLDER_ID_HERE' with your actual Google Drive Folder ID to save '{filename}'.")
            return None

        try:
            credentials = service_account.Credentials.from_service_account_file(
                    self.service_account_file,
                    scopes=self.scopes)

            drive_service = build('drive', 'v3', credentials=credentials)
            print(f"Google Drive API service built successfully for uploading '{filename}'.")

            if file_type.lower() == 'csv':
                buffer = io.StringIO()
                df_to_upload.to_csv(buffer, index=True)
                content = buffer.getvalue().encode('utf-8')
                mime_type = 'text/csv'
            elif file_type.lower() == 'xlsx':
                buffer = io.BytesIO()
                df_to_upload.to_excel(buffer, index=False) # Use to_excel
                content = buffer.getvalue()
                mime_type = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet' # XLSX MIME type
            else:
                print(f"Unsupported file type: {file_type}. Only 'csv' and 'xlsx' are supported for upload.")
                return None

            media_body = MediaIoBaseUpload(io.BytesIO(content),
                                           mimetype=mime_type,
                                           resumable=True)

            file_metadata = {
                'name': filename,
                'parents': [folder_id],
                'mimeType': mime_type
            }

            file = drive_service.files().create(
                body=file_metadata,
                media_body=media_body,
                fields='id',
                supportsAllDrives=True,
            ).execute()

            print(f"Successfully uploaded '{filename}' to Google Drive (File ID: {file.get('id')}).")
            return file.get('id')

        except FileNotFoundError:
            print(f"Error: Service account key file not found at '{self.service_account_file}'")
            print("Please ensure the path to your service account key is correct.")
            return None
        except HttpError as error:
            print(f"An HTTP error occurred during upload: {error}")
            return None
        except Exception as e:
            print(f"An unexpected error occurred during upload: {e}")
            return None

In [ ]:
# @title ## Extract File ID {"form-width":"20%"}
# @markdown This Python function is designed to extract the file ID from a Google Drive shared link.

import re

def extract_file_id(shared_link):
    """
    Extracts the Google Drive file ID from a shared link.

    Args:
        shared_link (str): The Google Drive shared link.

    Returns:
        str: The extracted file ID, or None if not found.
    """
    # Pattern 1: https://drive.google.com/file/d/FILE_ID/view
    # Pattern 2: https://drive.google.com/open?id=FILE_ID
    # Pattern 3: https://docs.google.com/spreadsheets/d/FILE_ID/edit (and similar for other doc types)
    # Pattern 4: https://drive.google.com/drive/folders/FOLDER_ID
    patterns = [
        r"https:\/\/drive\.google\.com\/file\/d\/([a-zA-Z0-9_-]+)",
        r"https:\/\/drive\.google\.com\/open\?id=([a-zA-Z0-9_-]+)",
        r"https:\/\/docs\.google\.com\/(?:spreadsheets|document|presentation)\/d\/([a-zA-Z0-9_-]+)",
        r"https:\/\/drive\.google\.com\/drive\/folders\/([a-zA-Z0-9_-]+)"
    ]

    for pattern in patterns:
        match = re.search(pattern, shared_link)
        if match:
            return match.group(1)

    return None

print("The 'extract_file_id' function has been defined.")

# Read in Data

In [ ]:
# @title Import Banner Roster xlsx files from a folder {"form-width":"20%"}
folder_link = "https://drive.google.com/drive/folders/1N2Xestj6Yx20ygbggBT6K7HFPNXUDRhP?usp=drive_link" # @param {"type":"string","placeholder":"Enter the Google Drive folder link containing the XLSX files"}

folder_id = extract_file_id(folder_link)

if folder_id is None:
    print("Error: Could not extract folder ID from the provided link.")
else:
    document_manager = GoogleDocumentManager()
    print(f"\nAttempting to read XLSX files from folder ID: {folder_id}...")

    # Google Drive MIME type for XLSX files
    xlsx_mime_type = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
    xlsx_files = document_manager.list_files_in_folder(folder_id, file_mime_type=xlsx_mime_type)

    all_dfs = []
    if xlsx_files:
        for file_info in xlsx_files:
            file_id = file_info['id']
            file_name = file_info['name']
            print(f"Reading file: {file_name} (ID: {file_id})")
            df_xlsx = document_manager.read_file(file_id, file_type='xlsx')

            if df_xlsx is not None:
                # Apply the same header processing logic as before
                if len(df_xlsx) > 13: # Ensure there are enough rows for the header
                    new_header = df_xlsx.iloc[13]
                    df_xlsx = df_xlsx[14:]
                    df_xlsx.columns = new_header
                    df_xlsx = df_xlsx.reset_index(drop=True)
                    print(f"Successfully processed '{file_name}'.")
                    all_dfs.append(df_xlsx)
                else:
                    print(f"Warning: '{file_name}' does not have enough rows for header processing. Skipping.")
            else:
                print(f"Failed to read '{file_name}'.")
    else:
        print("No XLSX files found in the specified folder.")

    if all_dfs:
        # Concatenate all DataFrames into a single one
        final_df_xlsx = pd.concat(all_dfs, ignore_index=True)
        print("\nCombined DataFrame head after processing all XLSX files:")
        display(final_df_xlsx.head())
        print(f"Combined DataFrame shape: {final_df_xlsx.shape}")
    else:
        final_df_xlsx = pd.DataFrame()
        print("No valid XLSX DataFrames were combined.")

In [ ]:
display(final_df_xlsx)


In [ ]:
# @title Save combined DataFrame to Google Drive as XLSX {"form-width":"20%"}
output_folder_link = "https://drive.google.com/drive/folders/1N2Xestj6Yx20ygbggBT6K7HFPNXUDRhP?usp=drive_link" # @param {"type":"string","placeholder":"Enter the Google Drive folder link where you want to save the XLSX file"}
output_filename = "Combined_Banner_Roster.xlsx" # @param {"type":"string"}

if 'final_df_xlsx' in locals() and not final_df_xlsx.empty:
    output_folder_id = extract_file_id(output_folder_link)

    if output_folder_id is None:
        print("Error: Could not extract output folder ID from the provided link. Please ensure the link is valid.")
    else:
        document_manager = GoogleDocumentManager()
        print(f"\nAttempting to upload '{output_filename}' to folder ID: {output_folder_id}...")
        uploaded_file_id = document_manager.upload_dataframe_to_drive(
            final_df_xlsx,
            output_filename,
            output_folder_id,
            file_type='xlsx' # Specify file type as xlsx
        )

        if uploaded_file_id:
            print(f"File uploaded successfully! New file ID: {uploaded_file_id}")
            print(f"You can view the file here: https://drive.google.com/file/d/{uploaded_file_id}/view")
        else:
            print("Failed to upload the DataFrame to Google Drive.")
else:
    print("No DataFrame named 'final_df_xlsx' found or it is empty. Please ensure the previous steps ran successfully to create 'final_df_xlsx'.")